# Notebook 01: Data Collection

## Purpose
Collect resolved prediction market data from Polymarket and Kalshi.
For each market we collect:
- Market metadata (question, category, volume, dates)
- Hourly price history (for time-series feature engineering in NB 02)

## Outputs
- `data/raw/markets_polymarket.parquet`
- `data/raw/prices_polymarket.parquet`
- `data/raw/markets_kalshi.parquet`
- `data/raw/prices_kalshi.parquet`

## APIs
- Polymarket Gamma API: gamma-api.polymarket.com
- Polymarket CLOB API: clob.polymarket.com
- Kalshi Trade API: external-api.kalshi.com/trade-api/v2

In [18]:
# Cell 2 — Imports and setup
import requests
import pandas as pd
import time
from pathlib import Path
from tqdm import tqdm

Path("../data/raw").mkdir(parents=True, exist_ok=True)
Path("../data/processed").mkdir(parents=True, exist_ok=True)

print("Ready.")

Ready.


In [19]:
# Cell 3 — Fetch resolved Polymarket events by category
# tag_id 2=Politics, 120=Finance, 757=Legal, 1396=International affairs
TARGET_TAGS = [2, 120, 757, 1396]

def fetch_polymarket_events(tag_id, max_pages=10, limit=50):
    base_url = "https://gamma-api.polymarket.com/events"
    all_events = []
    offset = 0

    for _ in range(max_pages):
        params = {
            "closed": "true",
            "tag_id": tag_id,
            "order": "volume",
            "ascending": "false",
            "limit": limit,
            "offset": offset
        }
        resp = requests.get(base_url, params=params)
        resp.raise_for_status()
        data = resp.json()
        if not data:
            break
        all_events.extend(data)
        if len(data) < limit:
            break
        offset += limit
        time.sleep(0.3)

    return all_events

all_events = []
for tag_id in TARGET_TAGS:
    events = fetch_polymarket_events(tag_id)
    print(f"tag_id={tag_id}: {len(events)} events")
    all_events.extend(events)

# Deduplicate by event id
seen = set()
unique_events = []
for e in all_events:
    if e["id"] not in seen:
        seen.add(e["id"])
        unique_events.append(e)

print(f"\nTotal unique events: {len(unique_events)}")

tag_id=2: 500 events
tag_id=120: 500 events
tag_id=757: 3 events
tag_id=1396: 1 events

Total unique events: 998


In [23]:
# Cell 4 — Extract markets from events, save market metadata
def parse_polymarket_market(market, event):
    token_ids = market.get("clobTokenIds", "[]")
    if isinstance(token_ids, str):
        import json
        token_ids = json.loads(token_ids)
    return {
        "market_id": market.get("conditionId"),
        "event_id": event.get("id"),
        "question": market.get("question"),
        "category": next((t["label"] for t in event.get("tags", [])
                         if str(t["id"]) in [str(x) for x in TARGET_TAGS]), None),
        "start_date": event.get("startDate"),
        "end_date": event.get("endDate"),
        "closed_time": event.get("closedTime"),
        "volume": market.get("volume"),
        "liquidity": market.get("liquidity"),
        "enable_order_book": market.get("enableOrderBook"),
        "token_ids": token_ids,
        "platform": "polymarket"
    }

pm_markets = []
for event in unique_events:
    for market in event.get("markets", []):
        pm_markets.append(parse_polymarket_market(market, event))

pm_markets_df = pd.DataFrame(pm_markets)
pm_markets_df["volume"] = pd.to_numeric(pm_markets_df["volume"], errors="coerce")
pm_markets_df["liquidity"] = pd.to_numeric(pm_markets_df["liquidity"], errors="coerce")

# Only keep markets with order book enabled (required for price history)
pm_markets_df = pm_markets_df[pm_markets_df["enable_order_book"] == True].copy()

pm_markets_df.to_parquet("../data/raw/markets_polymarket.parquet", index=False)
print(f"Saved {len(pm_markets_df)} Polymarket markets")
pm_markets_df.head()

Saved 10087 Polymarket markets


,market_id,event_id,question,category,start_date,end_date,closed_time,volume,liquidity,enable_order_book,token_ids,platform
0,0xdd22472e552920b8438158ea7238bfadfa4f736aa4ce...,903193,Will Donald Trump win the 2024 US Presidential...,Politics,2024-01-04T22:58:00Z,2024-11-05T12:00:00Z,2024-11-06T20:40:10Z,1.531479e+09,NaN,True,[217426331434639062905690501558262415330672727...,polymarket
1,0x14018049e265a2d88f284be9588e2e3542e3a3df08cc...,903193,Will Joe Biden win the 2024 US Presidential El...,Politics,2024-01-04T22:58:00Z,2024-11-05T12:00:00Z,2024-11-06T20:40:10Z,7.217611e+07,NaN,True,[880278396092436241934156141793286796026129164...,polymarket
2,0xced9f9d90c94db9f1e1dbd7d9fba82fe4fa7431c0d4e...,903193,Will Nikki Haley win the 2024 US Presidential ...,Politics,2024-01-04T22:58:00Z,2024-11-05T12:00:00Z,2024-11-06T20:40:10Z,1.075292e+08,NaN,True,[190833494627915933345328405488906021871857399...,polymarket
3,0x40bbdd26dc08406eedcb913efee7f7faddf50e16fc21...,903193,Will Gavin Newsom win the 2024 US Presidential...,Politics,2024-01-04T22:58:00Z,2024-11-05T12:00:00Z,2024-11-06T20:40:10Z,5.416128e+07,NaN,True,[992003473651697607003854531648781885044795484...,polymarket
4,0x7da35195ac3c7bf167f88ab0c27067a99020e36de67d...,903193,Will Robert F. Kennedy Jr. win the 2024 US Pre...,Politics,2024-01-04T22:58:00Z,2024-11-05T12:00:00Z,2024-11-06T20:40:10Z,1.416051e+08,NaN,True,[755518906810497964057762956544380997763335715...,polymarket


In [24]:
# Add this between Cell 4 and Cell 5
# Filter to high volume markets only and cap total
pm_markets_df["volume"] = pd.to_numeric(pm_markets_df["volume"], errors="coerce")
pm_markets_df = pm_markets_df[pm_markets_df["volume"] > 10000].copy()
pm_markets_df = pm_markets_df.sort_values("volume", ascending=False).head(15).copy()

print(f"Markets after filtering: {len(pm_markets_df)}")
print(f"Volume range: ${pm_markets_df['volume'].min():,.0f} to ${pm_markets_df['volume'].max():,.0f}")
pm_markets_df[["question", "category", "volume"]].head(10)

Markets after filtering: 15
Volume range: $141,605,111 to $1,531,479,285


,question,category,volume
0,Will Donald Trump win the 2024 US Presidential...,Politics,1.531479e+09
7,Will Kamala Harris win the 2024 US Presidentia...,Politics,1.037039e+09
144,Will Donald Trump be inaugurated?,Politics,4.004095e+08
13,Will any other Republican Politician win the 2...,Politics,2.416551e+08
17,Fed decreases interest rates by 50+ bps after ...,Politics,2.350652e+08
20,Fed increases interest rates by 25+ bps after ...,Politics,2.164557e+08
283,"US x Iran ceasefire extended by April 22, 2026?",Politics,2.036214e+08
226,US x Iran ceasefire by April 7?,Politics,1.736962e+08
232,Will the Fed increase interest rates by 25+ bp...,Politics,1.729721e+08
23,Kamala Harris wins the popular vote?,Politics,1.637798e+08


In [25]:
# Cell 5 — Fetch Polymarket price history using startTs/endTs chunked approach
from datetime import datetime, timedelta
import json

def fetch_price_history_chunked(token_id, start_date, end_date, chunk_days=15):
    """
    Fetch hourly price history for a token by chunking into date ranges.
    Uses startTs/endTs instead of interval=max which fails for resolved markets.
    """
    all_history = []
    current_start = start_date

    while current_start < end_date:
        current_end = min(current_start + timedelta(days=chunk_days), end_date)

        resp = requests.get(
            "https://clob.polymarket.com/prices-history",
            params={
                "market": token_id,
                "startTs": int(current_start.timestamp()),
                "endTs": int(current_end.timestamp()),
                "fidelity": 60
            }
        )
        if resp.status_code == 200:
            history = resp.json().get("history", [])
            all_history.extend(history)

        current_start = current_end
        time.sleep(0.2)

    return all_history

all_prices = []
skipped = 0

for _, row in tqdm(pm_markets_df.iterrows(), total=len(pm_markets_df), desc="Fetching PM prices"):
    market_id = row["market_id"]
    token_ids = row["token_ids"]

    if not token_ids:
        skipped += 1
        continue

    # Parse token_ids if still a string
    if isinstance(token_ids, str):
        token_ids = json.loads(token_ids)

    # Get date range from market metadata
    start_date = pd.to_datetime(row["start_date"], utc=True)
    end_date = pd.to_datetime(row["end_date"], utc=True)

    if pd.isna(start_date) or pd.isna(end_date):
        skipped += 1
        continue

    # Convert to naive datetime for timedelta arithmetic
    start_dt = start_date.to_pydatetime().replace(tzinfo=None)
    end_dt = end_date.to_pydatetime().replace(tzinfo=None)

    # Fetch YES token only (index 0)
    token_id = token_ids[0]
    history = fetch_price_history_chunked(token_id, start_dt, end_dt)

    for point in history:
        all_prices.append({
            "market_id": market_id,
            "token_id": token_id,
            "timestamp": point["t"],
            "price": point["p"]
        })

pm_prices_df = pd.DataFrame(all_prices)
if not pm_prices_df.empty:
    pm_prices_df["timestamp"] = pd.to_datetime(pm_prices_df["timestamp"], unit="s", utc=True)

pm_prices_df.to_parquet("../data/raw/prices_polymarket.parquet", index=False)
print(f"Saved {len(pm_prices_df):,} price points across {pm_prices_df['market_id'].nunique()} markets")
print(f"Skipped {skipped} markets (missing token IDs or dates)")
if not pm_prices_df.empty:
    print(f"Date range: {pm_prices_df['timestamp'].min()} to {pm_prices_df['timestamp'].max()}")

Fetching PM prices: 100%|██████████| 15/15 [01:56<00:00,  7.76s/it]

Saved 65,083 price points across 14 markets
Skipped 1 markets (missing token IDs or dates)
Date range: 2024-01-05 06:00:03+00:00 to 2026-04-22 06:00:13+00:00


In [26]:
print(pm_prices_df["market_id"].unique())
print("\nMarkets with price data:")
pm_markets_df[pm_markets_df["market_id"].isin(pm_prices_df["market_id"].unique())][["question", "category", "volume"]]

['0xdd22472e552920b8438158ea7238bfadfa4f736aa4cee91a6b86c39ead110917'
 '0xc6485bb7ea46d7bb89beb9c91e7572ecfc72a6273789496f78bc5e989e4d1638'
 '0xc3d4155148681756bfe67bb41d8d0882a8a122e7d3762b3591bf6598c9bd198b'
 '0x55c551896c10a74861f2fd88b4f928694310114704cc74b29b9760d1156cade6'
 '0x17815081230e3b9c78b098162c33b1ffa68c4ec29c123d3d14989599e0c2e113'
 '0x7c6c69d91b21cbbea08a13d0ad51c0e96a956045aaadc77bce507c6b0475b66e'
 '0x1d2787cb8aed975d092b2799ed6f4083e9445f7420cdc09e9d47e7d54356c6cd'
 '0x25aa90b3cd98305e849189b4e8b770fc77fe89bccb7cf9656468414e01145d38'
 '0x265366ede72d73e137b2b9095a6cdc9be6149290caa295738a95e3d881ad0865'
 '0x6903b766f5fda3d5b02f4472a6b4154419e78b7fd126c0b29ce17bb8e20b20cc'
 '0x43ec78527bd98a0588dd9455685b2cc82f5743140cb3a154603dc03c02b57de5'
 '0x230144e34a84dfd0ebdc6de7fde37780e28154f6f84dd8880c7f0e58d302d448'
 '0xebddfcf7b4401dade8b4031770a1ab942b01854f3bed453d5df9425cd9f211a9'
 '0x7da35195ac3c7bf167f88ab0c27067a99020e36de67d39968b71d9debcdd925e']

Markets with price

,question,category,volume
0,Will Donald Trump win the 2024 US Presidential...,Politics,1.531479e+09
7,Will Kamala Harris win the 2024 US Presidentia...,Politics,1.037039e+09
144,Will Donald Trump be inaugurated?,Politics,4.004095e+08
13,Will any other Republican Politician win the 2...,Politics,2.416551e+08
17,Fed decreases interest rates by 50+ bps after ...,Politics,2.350652e+08
20,Fed increases interest rates by 25+ bps after ...,Politics,2.164557e+08
283,"US x Iran ceasefire extended by April 22, 2026?",Politics,2.036214e+08
232,Will the Fed increase interest rates by 25+ bp...,Politics,1.729721e+08
23,Kamala Harris wins the popular vote?,Politics,1.637798e+08
164,Fed decreases interest rates by 50+ bps after ...,Politics,1.615744e+08


In [31]:
# Cell 6 — Fetch Kalshi markets from both live and historical tiers
import time as time_module

KALSHI_SERIES = ["KXFED", "KXGDP", "FED", "KXINFL", "KXUNEMP"]
HISTORICAL_CUTOFF = pd.Timestamp("2026-04-02", tz="UTC")

def fetch_kalshi_historical_markets(series_ticker, max_pages=10):
    all_markets = []
    cursor = None
    for _ in range(max_pages):
        params = {"limit": 100, "series_ticker": series_ticker}
        if cursor:
            params["cursor"] = cursor
        resp = requests.get(
            "https://external-api.kalshi.com/trade-api/v2/historical/markets",
            params=params
        )
        if resp.status_code != 200:
            break
        data = resp.json()
        markets = data.get("markets", [])
        all_markets.extend(markets)
        cursor = data.get("cursor")
        if not cursor:
            break
        time.sleep(0.3)
    return all_markets

def fetch_kalshi_live_markets(series_ticker, max_pages=5):
    all_markets = []
    cursor = None
    for _ in range(max_pages):
        params = {"limit": 100, "series_ticker": series_ticker, "status": "settled"}
        if cursor:
            params["cursor"] = cursor
        resp = requests.get(
            "https://external-api.kalshi.com/trade-api/v2/markets",
            params=params
        )
        if resp.status_code != 200:
            break
        data = resp.json()
        markets = data.get("markets", [])
        all_markets.extend(markets)
        cursor = data.get("cursor")
        if not cursor:
            break
        time.sleep(0.3)
    return all_markets

kalshi_raw = []
for series in KALSHI_SERIES:
    hist = fetch_kalshi_historical_markets(series)
    live = fetch_kalshi_live_markets(series)
    print(f"{series}: {len(hist)} historical + {len(live)} live")
    kalshi_raw.extend(hist)
    kalshi_raw.extend(live)

# Deduplicate by ticker
seen = set()
unique_raw = []
for m in kalshi_raw:
    t = m.get("ticker")
    if t not in seen:
        seen.add(t)
        unique_raw.append(m)

print(f"\nTotal unique Kalshi markets: {len(unique_raw)}")
kalshi_raw = unique_raw

KXFED: 369 historical + 11 live
KXGDP: 166 historical + 8 live
FED: 0 historical + 0 live
KXINFL: 0 historical + 0 live
KXUNEMP: 0 historical + 0 live

Total unique Kalshi markets: 554


In [32]:
# Cell 7 — Parse and filter Kalshi markets
def get_series_ticker(ticker):
    parts = ticker.split("-")
    return parts[0] if parts else ticker

def parse_kalshi_market(m):
    ticker = m.get("ticker", "")
    # historical markets use settlement_ts, live use close_time
    end_date = m.get("settlement_ts") or m.get("close_time")
    start_date = m.get("open_time")
    return {
        "market_id": ticker,
        "series_ticker": get_series_ticker(ticker),
        "question": m.get("title"),
        "category": m.get("event_ticker"),
        "start_date": start_date,
        "end_date": end_date,
        "volume": m.get("volume_fp") or m.get("volume"),
        "liquidity": m.get("liquidity_dollars"),
        "outcome": m.get("result"),
        "platform": "kalshi"
    }

kalshi_markets_df = pd.DataFrame([parse_kalshi_market(m) for m in kalshi_raw])
kalshi_markets_df["volume"] = pd.to_numeric(kalshi_markets_df["volume"], errors="coerce")
kalshi_markets_df["liquidity"] = pd.to_numeric(kalshi_markets_df["liquidity"], errors="coerce")
kalshi_markets_df["end_date"] = pd.to_datetime(kalshi_markets_df["end_date"], utc=True, errors="coerce")
kalshi_markets_df["start_date"] = pd.to_datetime(kalshi_markets_df["start_date"], utc=True, errors="coerce")

# Exclude parlay markets and filter to meaningful volume
kalshi_markets_df = kalshi_markets_df[
    ~kalshi_markets_df["market_id"].str.startswith("KXMV")
].copy()
kalshi_markets_df = kalshi_markets_df[kalshi_markets_df["volume"] > 1000].copy()

# Flag which tier each market belongs to
kalshi_markets_df["is_historical"] = kalshi_markets_df["end_date"] < HISTORICAL_CUTOFF

kalshi_markets_df.to_parquet("../data/raw/markets_kalshi.parquet", index=False)
print(f"Saved {len(kalshi_markets_df)} Kalshi markets after filtering")
print(f"  Historical tier: {kalshi_markets_df['is_historical'].sum()}")
print(f"  Live tier: {(~kalshi_markets_df['is_historical']).sum()}")
kalshi_markets_df.head()

Saved 472 Kalshi markets after filtering
  Historical tier: 453
  Live tier: 19


,market_id,series_ticker,question,category,start_date,end_date,volume,liquidity,outcome,platform,is_historical
0,KXFED-26MAR-T5.25,KXFED,Will the upper bound of the federal funds rate...,KXFED-26MAR,2025-08-06 14:00:00+00:00,2026-03-18 18:13:09.227730+00:00,4641.0,0.0,no,kalshi,True
1,KXFED-26MAR-T5.00,KXFED,Will the upper bound of the federal funds rate...,KXFED-26MAR,2025-08-06 14:00:00+00:00,2026-03-18 18:13:09.227730+00:00,1880.0,0.0,no,kalshi,True
2,KXFED-26MAR-T4.75,KXFED,Will the upper bound of the federal funds rate...,KXFED-26MAR,2025-08-06 14:00:00+00:00,2026-03-18 18:13:09.227730+00:00,6652.0,0.0,no,kalshi,True
3,KXFED-26MAR-T4.50,KXFED,Will the upper bound of the federal funds rate...,KXFED-26MAR,2025-08-06 14:00:00+00:00,2026-03-18 18:13:09.227730+00:00,10659.0,0.0,no,kalshi,True
4,KXFED-26MAR-T4.25,KXFED,Will the upper bound of the federal funds rate...,KXFED-26MAR,2025-08-06 14:00:00+00:00,2026-03-18 18:13:09.227730+00:00,22014.0,0.0,no,kalshi,True


In [34]:
# Cell 8 — Fetch Kalshi candlesticks (multithreaded)
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import timedelta

def fetch_kalshi_candlesticks(row):
    ticker = row["market_id"]
    series_ticker = row["series_ticker"]
    is_historical = row["is_historical"]
    start_date = row["start_date"]
    end_date = row["end_date"]

    if pd.isna(start_date) or pd.isna(end_date):
        return ticker, []

    start_ts = int(start_date.timestamp())
    end_ts = int(end_date.timestamp())

    chunk_seconds = 200 * 24 * 3600
    all_candles = []
    current_start = start_ts

    while current_start < end_ts:
        current_end = min(current_start + chunk_seconds, end_ts)

        if is_historical:
            url = f"https://external-api.kalshi.com/trade-api/v2/historical/markets/{ticker}/candlesticks"
        else:
            url = f"https://external-api.kalshi.com/trade-api/v2/series/{series_ticker}/markets/{ticker}/candlesticks"

        try:
            resp = requests.get(url, params={
                "start_ts": current_start,
                "end_ts": current_end,
                "period_interval": 60
            }, timeout=10)
            if resp.status_code == 200:
                candles = resp.json().get("candlesticks", [])
                all_candles.extend(candles)
        except Exception:
            pass

        current_start = current_end

    return ticker, all_candles

all_kalshi_prices = []
rows = [row for _, row in kalshi_markets_df.iterrows()]

with ThreadPoolExecutor(max_workers=5) as executor:
    futures = {executor.submit(fetch_kalshi_candlesticks, row): row["market_id"] for row in rows}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Fetching Kalshi candlesticks"):
        ticker, candles = future.result()
        for c in candles:
            all_kalshi_prices.append({
                "market_id": ticker,
                "timestamp": c.get("end_period_ts"),
                "price_open": c.get("price", {}).get("open"),
                "price_close": c.get("price", {}).get("close"),
                "price_high": c.get("price", {}).get("high"),
                "price_low": c.get("price", {}).get("low"),
                "price_mean": c.get("price", {}).get("mean"),
                "volume": c.get("volume"),
                "open_interest": c.get("open_interest"),
            })

kalshi_prices_df = pd.DataFrame(all_kalshi_prices)
if not kalshi_prices_df.empty:
    kalshi_prices_df["timestamp"] = pd.to_datetime(
        kalshi_prices_df["timestamp"], unit="s", utc=True
    )
    for col in ["price_open", "price_close", "price_high", "price_low", "price_mean", "volume", "open_interest"]:
        kalshi_prices_df[col] = pd.to_numeric(kalshi_prices_df[col], errors="coerce")

kalshi_prices_df.to_parquet("../data/raw/prices_kalshi.parquet", index=False)
print(f"Saved {len(kalshi_prices_df)} Kalshi price points across {kalshi_prices_df['market_id'].nunique()} markets")

Fetching Kalshi candlesticks: 100%|██████████| 472/472 [01:10<00:00,  6.68it/s]


Saved 266124 Kalshi price points across 217 markets


In [35]:
# Cell 9 — Sanity check
print("=== Collection Summary ===")
print(f"Polymarket markets:      {len(pm_markets_df):,}")
print(f"Polymarket price points: {len(pm_prices_df):,}")
print(f"Kalshi markets:          {len(kalshi_markets_df):,}")
print(f"Kalshi price points:     {len(kalshi_prices_df):,}")

print("\nPolymarket price date range:")
if not pm_prices_df.empty:
    print(f"  {pm_prices_df['timestamp'].min()} to {pm_prices_df['timestamp'].max()}")

print("\nKalshi price date range:")
if not kalshi_prices_df.empty:
    print(f"  {kalshi_prices_df['timestamp'].min()} to {kalshi_prices_df['timestamp'].max()}")

print("\nPolymarket categories:")
print(pm_markets_df["category"].value_counts())

print("\nSample Polymarket markets:")
print(pm_markets_df[["question", "category", "volume"]].head(5))

=== Collection Summary ===
Polymarket markets:      15
Polymarket price points: 65,083
Kalshi markets:          472
Kalshi price points:     266,124

Polymarket price date range:
  2024-01-05 06:00:03+00:00 to 2026-04-22 06:00:13+00:00

Kalshi price date range:
  2021-07-02 15:00:00+00:00 to 2026-04-30 13:00:00+00:00

Polymarket categories:
category
Politics    15
Name: count, dtype: int64

Sample Polymarket markets:
                                              question  category        volume
0    Will Donald Trump win the 2024 US Presidential...  Politics  1.531479e+09
7    Will Kamala Harris win the 2024 US Presidentia...  Politics  1.037039e+09
144                  Will Donald Trump be inaugurated?  Politics  4.004095e+08
13   Will any other Republican Politician win the 2...  Politics  2.416551e+08
17   Fed decreases interest rates by 50+ bps after ...  Politics  2.350652e+08
